# S1-3. 다중 에이전트 강화학습 (온라인 보충)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/online/%EC%98%A8%EB%9D%BC%EC%9D%B8%EB%B3%B4%EC%B6%A9_S1-3_%EB%8B%A4%EC%A4%91_%EC%97%90%EC%9D%B4%EC%A0%84%ED%8A%B8_%EA%B0%95%ED%99%94%ED%95%99%EC%8A%B5.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## 실습: 협력 격자에서 VDN

### 🔬 실습 — 2-에이전트 VDN 학습

① **에이전트별 Q-network**: 각자 로컬 관측을 받아 행동별 Q값 출력.


In [ ]:
import torch, torch.nn as nn

class AgentQNet(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, 64), nn.ReLU(),
                                 nn.Linear(64, action_dim))
    def forward(self, o): return self.net(o)

N = 2                                          # 에이전트 수
agents = [AgentQNet(obs_dim=6, action_dim=4) for _ in range(N)]
targets = [AgentQNet(6, 4) for _ in range(N)]
for a, t in zip(agents, targets): t.load_state_dict(a.state_dict())

② **VDN 팀 Q 분해**: 개별 Q를 합쳐 팀 Q를 만든다.


In [ ]:
def q_tot(nets, obs_list, act_list):
    # obs_list[i]:(B,obs), act_list[i]:(B,)  ->  Q_tot:(B,)
    q = 0
    for i, net in enumerate(nets):
        qi = net(obs_list[i]).gather(1, act_list[i].unsqueeze(1)).squeeze(1)
        q = q + qi                              # Q_tot = sum_i Q^ i
    return q

③ **팀 TD 타깃**: 다음 상태에서 각자 최대 Q의 합(IGM 성질).


In [ ]:
def team_target(targets, next_obs_list, r_tot, done, gamma=0.99):
    with torch.no_grad():
        max_sum = 0
        for i, net in enumerate(targets):
            max_sum = max_sum + net(next_obs_list[i]).max(1).values
        return r_tot + gamma * max_sum * (1 - done)

④ **학습 스텝**: 팀 손실 하나로 모든 에이전트 Q를 동시에 갱신.


In [ ]:
import torch.optim as optim
params = [p for net in agents for p in net.parameters()]
optimizer = optim.Adam(params, lr=1e-3)

def train_step(batch):
    obs, act, r_tot, nobs, done = batch         # 공유 리플레이 버퍼에서
    q = q_tot(agents, obs, act)
    y = team_target(targets, nobs, r_tot, done)
    loss = nn.functional.smooth_l1_loss(q, y)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    return loss.item()

### 🔬 실습 — IQL vs VDN 대조 실험: "팀으로 배우기"의 효과를 수치로

① **두 학습 방식의 손실만 다르게**: 나머지(버퍼·타깃·탐색)는 동일하게 둔다.


> ⚠️ **설명용 조각** — 아래 두 손실 함수는 IQL과 VDN의 차이를 드러내는 의사코드입니다. `q1_tgt`, `q2_tgt`, `gamma`, `F`, `batch` 등은 이 노트북에 정의되어 있지 않으므로 함수를 호출하면 오류가 납니다(정의만 하는 이 셀은 그대로 실행됩니다).

In [ ]:
# IQL: 에이전트별 '개별' TD 손실 (남의 학습을 모른 채 각자 갱신)
def iql_loss(q1, q2, batch):
    y1 = batch.r + gamma * (1 - batch.d) * q1_tgt(batch.o1_next).max(1).values
    y2 = batch.r + gamma * (1 - batch.d) * q2_tgt(batch.o2_next).max(1).values
    l1 = F.smooth_l1_loss(q1(batch.o1).gather(1, batch.a1), y1.unsqueeze(1))
    l2 = F.smooth_l1_loss(q2(batch.o2).gather(1, batch.a2), y2.unsqueeze(1))
    return l1 + l2          # 각자 자기 목표를 좇는다

# VDN: 개별 Q를 '합쳐' 하나의 팀 TD 손실 (팀 보상 하나로 함께 갱신)
def vdn_loss(q1, q2, batch):
    q_tot = (q1(batch.o1).gather(1, batch.a1)
             + q2(batch.o2).gather(1, batch.a2)).squeeze(1)
    with torch.no_grad():
        y = batch.r + gamma * (1 - batch.d) * (
            q1_tgt(batch.o1_next).max(1).values
            + q2_tgt(batch.o2_next).max(1).values)
    return F.smooth_l1_loss(q_tot, y)   # 팀 전체가 하나의 목표를 좇는다

---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
